[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/three_point_pose/three_point_pose.ipynb)

# Recovering a pose from three markers

Three measured markers locate a rigid object. Match one point, then the line through two points, then the plane through all three: each step removes the freedom left by the previous one. Orange marks the measured target; teal follows the reconstructed pose.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D
from examples.animation import save_figure, show_animation
from examples.geometry.three_point_pose import render

In [ ]:
ga = PGA3D
mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
Line = ga.gatype.bivector()
Plane = ga.gatype.vector()
Motor = ga.gatype.rotor()

In [ ]:
source: Point = (mv("x y z", [[-0.8, -0.4, 0.0], [0.9, -0.3, 0.0], [-0.3, 0.8, 0.2]]) + mv.w).dual()
displacement: Line = mv.xw * 0.7 + mv.yw * 0.2 + mv.zw * 0.4
rotation: Line = mv.xy * 0.6 + mv.yz * 0.2

# A known motion supplies exact measurements; the reconstruction sees only the points.
pose: Motor = displacement.exp() * rotation.exp()
target: Point = pose >> source
save_figure(render.draw_pose(source, target, source), "three_point_pose_measured");

In [ ]:
# The ratio gives twice the translation between the first corresponding points.
translation: Motor = (target[0] / source[0]).square_root()
translated: Point = translation >> source
save_figure(render.draw_pose(source, target, translated), "three_point_pose_point");

In [ ]:
# Rotate the joining line onto its target. Their common point stays fixed.
target_line: Line = (target[0] & target[1]).normalized()
current_line: Line = (target[0] & translated[1]).normalized()
turn: Motor = (target_line / current_line).square_root()
aligned: Point = turn >> translated
save_figure(render.draw_pose(source, target, aligned), "three_point_pose_line");

In [ ]:
# Matching the two marker planes removes the remaining roll about that line.
target_plane: Plane = (target_line & target[2]).normalized()
current_plane: Plane = (target_line & aligned[2]).normalized()
roll: Motor = (target_plane / current_plane).square_root()
reconstructed: Motor = roll * turn * translation
restored: Point = reconstructed >> source
save_figure(render.draw_pose(source, target, restored), "three_point_pose_plane");

In [ ]:
# Undoing the pose that made the measurements leaves the identity motor.
residual: Motor = reconstructed * pose.reverse()

print(residual.coefficient_table())

The same square-root construction works for points, lines and planes. This exact construction assumes congruent, noncollinear marker triples and avoids opposing oriented elements at the intermediate steps. Measurements with noise need a fitting criterion; matching these three constraints sequentially is not a least-squares fit.

In [ ]:
steps = 24
duration_ms = 65
fractions = np.linspace(0, 1, steps + 1)[1:]
fractions = fractions * fractions * (3 - 2 * fractions)
increments: Motor = stack([translation, turn, roll])


def alignments() -> Iterator[Point]:
    placed = source
    yield placed
    for increment in increments:
        start = placed
        for fraction in fractions:
            # A fraction of the motor's logarithm moves the triangle rigidly, at a constant rate along its screw.
            placed = (increment.log() * fraction).exp() >> start
            yield placed


show_animation(render.animate(alignments(), source, target), "three_point_pose", duration_ms)

Construction adapted from ganja.js’s [motor reconstruction example](https://github.com/enkimute/ganja.js/blob/master/examples/example_chapter11_motor_reconstruction.html).